In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/dataset.zip" -d "/content/"

In [ ]:
!unzip -q "/content/drive/MyDrive/generalization_test.zip" -d "/content/"

In [ ]:
!unzip -q "/content/drive/MyDrive/ndataset.zip" -d "/content/"

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class ConstellationDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)

        self.df.columns = self.df.columns.str.strip()

        self.img_dir = img_dir
        self.transform = transform

        self.df['mod_idx'] = self.df['modulation'].astype('category').cat.codes
        self.df['pn_idx'] = self.df['phase_noise'].astype('category').cat.codes
        self.df['iq_idx'] = self.df['iq_imbalance'].astype('category').cat.codes
        self.df['jam_idx'] = self.df['jamming'].astype('category').cat.codes
        self.df['snr_idx'] = self.df['snr_range'].astype('category').cat.codes

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('L')

        if self.transform:
            image = self.transform(image)

        labels = {
            'mod': torch.tensor(self.df.iloc[idx]['mod_idx'], dtype=torch.long),
            'pn': torch.tensor(self.df.iloc[idx]['pn_idx'], dtype=torch.long),
            'iq': torch.tensor(self.df.iloc[idx]['iq_idx'], dtype=torch.long),
            'jam': torch.tensor(self.df.iloc[idx]['jam_idx'], dtype=torch.long),
            'snr': torch.tensor(self.df.iloc[idx]['snr_idx'], dtype=torch.long)
        }
        return image, labels

my_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

my_dataset = ConstellationDataset('/content/labels.csv', '/content/images', my_transforms)
print(f"Φορτώθηκαν {len(my_dataset)} εικόνες.")

In [ ]:
from torch.utils.data import random_split, DataLoader

total_size = len(my_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    my_dataset, [train_size, val_size, test_size], generator=generator
)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


print(f"Train: {train_size}")
print(f"Validation: {val_size}")
print(f"Test: {test_size}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class MultiTaskLeNet_PAK(nn.Module):
    def __init__(self, num_mod=16, num_pn=4, num_iq=4, num_jam=4, num_snr=8):
        super(MultiTaskLeNet_PAK, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=(5, 5))
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=(3, 6))
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten_size = 16 * 62 * 60

        self.fc1 = nn.Linear(self.flatten_size, 120)
        self.fc2 = nn.Linear(120, 84)

        self.head_mod = nn.Linear(84, num_mod)
        self.head_pn = nn.Linear(84, num_pn)
        self.head_iq = nn.Linear(84, num_iq)
        self.head_jam = nn.Linear(84, num_jam)
        self.head_snr = nn.Linear(84, num_snr)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.flatten_size) # "Ξεδιπλώνει" τον πίνακα σε ευθεία γραμμή
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.head_mod(x), self.head_pn(x), self.head_iq(x), self.head_jam(x), self.head_snr(x)



class MultiTaskResNet18(nn.Module):
    def __init__(self, num_mod=16, num_pn=4, num_iq=4, num_jam=4, num_snr=8):
        super(MultiTaskResNet18, self).__init__()
        self.resnet = models.resnet18(weights=None)

        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        self.dropout = nn.Dropout(p=0.5)

        self.head_mod = nn.Linear(num_features, num_mod)
        self.head_pn = nn.Linear(num_features, num_pn)
        self.head_iq = nn.Linear(num_features, num_iq)
        self.head_jam = nn.Linear(num_features, num_jam)
        self.head_snr = nn.Linear(num_features, num_snr)

    def forward(self, x):
        features = self.resnet(x)
        features = self.dropout(features)

        return self.head_mod(features), self.head_pn(features), self.head_iq(features), self.head_jam(features), self.head_snr(features)

print("ResNet-18")



In [ ]:
import torch
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskLeNet_PAK().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


num_epochs = 15

lenet_train_losses = []
lenet_val_losses = []

for epoch in range(num_epochs):

    model.train()
    running_train_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        lbl_mod, lbl_pn = labels['mod'].to(device), labels['pn'].to(device)
        lbl_iq, lbl_jam = labels['iq'].to(device), labels['jam'].to(device)
        lbl_snr = labels['snr'].to(device)

        optimizer.zero_grad()

        out_mod, out_pn, out_iq, out_jam, out_snr = model(images)

        loss_mod = criterion(out_mod, lbl_mod)
        loss_pn = criterion(out_pn, lbl_pn)
        loss_iq = criterion(out_iq, lbl_iq)
        loss_jam = criterion(out_jam, lbl_jam)
        loss_snr = criterion(out_snr, lbl_snr)

        total_loss = loss_mod + loss_pn + loss_iq + loss_jam + loss_snr
        total_loss.backward()
        optimizer.step()

        running_train_loss += total_loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)
    lenet_train_losses.append(epoch_train_loss)


    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            lbl_mod, lbl_pn = labels['mod'].to(device), labels['pn'].to(device)
            lbl_iq, lbl_jam = labels['iq'].to(device), labels['jam'].to(device)
            lbl_snr = labels['snr'].to(device)

            out_mod, out_pn, out_iq, out_jam, out_snr = model(images)

            loss_mod = criterion(out_mod, lbl_mod)
            loss_pn = criterion(out_pn, lbl_pn)
            loss_iq = criterion(out_iq, lbl_iq)
            loss_jam = criterion(out_jam, lbl_jam)
            loss_snr = criterion(out_snr, lbl_snr)

            val_loss = loss_mod + loss_pn + loss_iq + loss_jam + loss_snr
            running_val_loss += val_loss.item()

    epoch_val_loss = running_val_loss / len(val_loader)
    lenet_val_losses.append(epoch_val_loss)

    print(f"Εποχή [{epoch+1}/{num_epochs}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")



In [ ]:
model.eval()

correct_mod = 0
correct_pn = 0
correct_iq = 0
correct_jam = 0
correct_snr = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        l_mod = labels['mod'].to(device)
        l_pn = labels['pn'].to(device)
        l_iq = labels['iq'].to(device)
        l_jam = labels['jam'].to(device)
        l_snr = labels['snr'].to(device)

        p_mod, p_pn, p_iq, p_jam, p_snr = model(images)

        _, pred_mod = torch.max(p_mod, 1)
        _, pred_pn = torch.max(p_pn, 1)
        _, pred_iq = torch.max(p_iq, 1)
        _, pred_jam = torch.max(p_jam, 1)
        _, pred_snr = torch.max(p_snr, 1)

        total += l_mod.size(0)
        correct_mod += (pred_mod == l_mod).sum().item()
        correct_pn += (pred_pn == l_pn).sum().item()
        correct_iq += (pred_iq == l_iq).sum().item()
        correct_jam += (pred_jam == l_jam).sum().item()
        correct_snr += (pred_snr == l_snr).sum().item()

print(f"Ακρίβεια Modulation:     {(100 * correct_mod / total):.2f}%")
print(f"Ακρίβεια Phase Noise:    {(100 * correct_pn / total):.2f}%")
print(f"Ακρίβεια I/Q Imbalance:  {(100 * correct_iq / total):.2f}%")
print(f"Ακρίβεια Jamming:        {(100 * correct_jam / total):.2f}%")
print(f"Ακρίβεια SNR Range:      {(100 * correct_snr / total):.2f}%")


In [ ]:
import torch


resnet_model = MultiTaskResNet18().to(device)

MODEL_PATH = "/content/drive/MyDrive/best_resnet_model.pth"

checkpoint = torch.load(MODEL_PATH, map_location=device)
resnet_model.load_state_dict(checkpoint)

resnet_model.to(device)
resnet_model.eval()

print(f"Το μοντέλο φορτώθηκε επιτυχώς από το αρχείο: {MODEL_PATH}")

correct_mod, correct_pn, correct_iq, correct_jam, correct_snr = 0, 0, 0, 0, 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        l_mod = labels['mod'].to(device)
        l_pn = labels['pn'].to(device)
        l_iq = labels['iq'].to(device)
        l_jam = labels['jam'].to(device)
        l_snr = labels['snr'].to(device)

        p_mod, p_pn, p_iq, p_jam, p_snr = resnet_model(images)

        _, pred_mod = torch.max(p_mod, 1)
        _, pred_pn = torch.max(p_pn, 1)
        _, pred_iq = torch.max(p_iq, 1)
        _, pred_jam = torch.max(p_jam, 1)
        _, pred_snr = torch.max(p_snr, 1)

        total += l_mod.size(0)
        correct_mod += (pred_mod == l_mod).sum().item()
        correct_pn += (pred_pn == l_pn).sum().item()
        correct_iq += (pred_iq == l_iq).sum().item()
        correct_jam += (pred_jam == l_jam).sum().item()
        correct_snr += (pred_snr == l_snr).sum().item()


print(f"Ακρίβεια Modulation:     {(100 * correct_mod / total):.2f}%")
print(f"Ακρίβεια Phase Noise:    {(100 * correct_pn / total):.2f}%")
print(f"Ακρίβεια I/Q Imbalance:  {(100 * correct_iq / total):.2f}%")
print(f"Ακρίβεια Jamming:        {(100 * correct_jam / total):.2f}%")
print(f"Ακρίβεια SNR Range:      {(100 * correct_snr / total):.2f}%")


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

model.eval()

all_true = {'mod': [], 'pn': [], 'iq': [], 'jam': [], 'snr': []}
all_pred = {'mod': [], 'pn': [], 'iq': [], 'jam': [], 'snr': []}

print("Συλλογή προβλέψεων από το Test Set...")

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        p_mod, p_pn, p_iq, p_jam, p_snr = model(images)

        _, pred_mod = torch.max(p_mod, 1)
        _, pred_pn = torch.max(p_pn, 1)
        _, pred_iq = torch.max(p_iq, 1)
        _, pred_jam = torch.max(p_jam, 1)
        _, pred_snr = torch.max(p_snr, 1)

        all_true['mod'].extend(labels['mod'].numpy())
        all_pred['mod'].extend(pred_mod.cpu().numpy())

        all_true['pn'].extend(labels['pn'].numpy())
        all_pred['pn'].extend(pred_pn.cpu().numpy())

        all_true['iq'].extend(labels['iq'].numpy())
        all_pred['iq'].extend(pred_iq.cpu().numpy())

        all_true['jam'].extend(labels['jam'].numpy())
        all_pred['jam'].extend(pred_jam.cpu().numpy())

        all_true['snr'].extend(labels['snr'].numpy())
        all_pred['snr'].extend(pred_snr.cpu().numpy())

class_names = {
    'mod': my_dataset.df['modulation'].astype('category').cat.categories.tolist(),
    'pn': my_dataset.df['phase_noise'].astype('category').cat.categories.tolist(),
    'iq': my_dataset.df['iq_imbalance'].astype('category').cat.categories.tolist(),
    'jam': my_dataset.df['jamming'].astype('category').cat.categories.tolist(),
    'snr': my_dataset.df['snr_range'].astype('category').cat.categories.tolist()
}

titles = {
    'mod': 'Confusion Matrix: Modulation',
    'pn': 'Confusion Matrix: Phase Noise',
    'iq': 'Confusion Matrix: I/Q Imbalance',
    'jam': 'Confusion Matrix: Jamming',
    'snr': 'Confusion Matrix: SNR Range'
}

print("Σχεδιασμός Πινάκων Σύγχυσης...\n")

for task in ['mod', 'pn', 'iq', 'jam', 'snr']:
    cm = confusion_matrix(all_true[task], all_pred[task])

    plt.figure(figsize=(10, 8))

    annot_kws = {"size": 8} if task == 'mod' else {"size": 12}

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names[task],
                yticklabels=class_names[task],
                annot_kws=annot_kws,
                cbar=False)

    plt.title(titles[task], fontsize=16, fontweight='bold', pad=15)
    plt.ylabel('True Label', fontsize=14, labelpad=10)
    plt.xlabel('Predicted Label', fontsize=14, labelpad=10)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("Φόρτωση των βαρών του καλύτερου μοντέλου (best_resnet_model.pth)...")
resnet_model.load_state_dict(torch.load('/content/drive/MyDrive/best_resnet_model.pth', map_location=device))

resnet_model.eval()

all_true_res = {'mod': [], 'pn': [], 'iq': [], 'jam': [], 'snr': []}
all_pred_res = {'mod': [], 'pn': [], 'iq': [], 'jam': [], 'snr': []}

print("Συλλογή προβλέψεων από το Test Set για τη ResNet-18...")

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        p_mod, p_pn, p_iq, p_jam, p_snr = resnet_model(images)

        _, pred_mod = torch.max(p_mod, 1)
        _, pred_pn = torch.max(p_pn, 1)
        _, pred_iq = torch.max(p_iq, 1)
        _, pred_jam = torch.max(p_jam, 1)
        _, pred_snr = torch.max(p_snr, 1)

        all_true_res['mod'].extend(labels['mod'].numpy())
        all_pred_res['mod'].extend(pred_mod.cpu().numpy())

        all_true_res['pn'].extend(labels['pn'].numpy())
        all_pred_res['pn'].extend(pred_pn.cpu().numpy())

        all_true_res['iq'].extend(labels['iq'].numpy())
        all_pred_res['iq'].extend(pred_iq.cpu().numpy())

        all_true_res['jam'].extend(labels['jam'].numpy())
        all_pred_res['jam'].extend(pred_jam.cpu().numpy())

        all_true_res['snr'].extend(labels['snr'].numpy())
        all_pred_res['snr'].extend(pred_snr.cpu().numpy())

class_names = {
    'mod': my_dataset.df['modulation'].astype('category').cat.categories.tolist(),
    'pn': my_dataset.df['phase_noise'].astype('category').cat.categories.tolist(),
    'iq': my_dataset.df['iq_imbalance'].astype('category').cat.categories.tolist(),
    'jam': my_dataset.df['jamming'].astype('category').cat.categories.tolist(),
    'snr': my_dataset.df['snr_range'].astype('category').cat.categories.tolist()
}

titles_res = {
    'mod': 'ResNet-18 Confusion Matrix: Modulation',
    'pn': 'ResNet-18 Confusion Matrix: Phase Noise',
    'iq': 'ResNet-18 Confusion Matrix: I/Q Imbalance',
    'jam': 'ResNet-18 Confusion Matrix: Jamming',
    'snr': 'ResNet-18 Confusion Matrix: SNR Range'
}

print("Σχεδιασμός Πινάκων Σύγχυσης...\n")

for task in ['mod', 'pn', 'iq', 'jam', 'snr']:
    cm = confusion_matrix(all_true_res[task], all_pred_res[task])

    plt.figure(figsize=(10, 8))

    annot_kws = {"size": 8} if task == 'mod' else {"size": 12}

    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=class_names[task],
                yticklabels=class_names[task],
                annot_kws=annot_kws,
                cbar=False)

    plt.title(titles_res[task], fontsize=16, fontweight='bold', pad=15)
    plt.ylabel('True Label', fontsize=14, labelpad=10)
    plt.xlabel('Predicted Label', fontsize=14, labelpad=10)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

resnet_model = MultiTaskResNet18().to(device)

print("1. Φόρτωση δεδομένων από το CSV...")
df = pd.read_csv('/content/ndataset/labels.csv')


model.to(device) # LeNet
model.eval()

resnet_model.load_state_dict(torch.load("/content/drive/MyDrive/best_resnet_model.pth", map_location=device))
resnet_model.to(device)
resnet_model.eval()

tasks = ['mod', 'pn', 'iq', 'jam', 'snr']

all_true = {task: [] for task in tasks}
all_pred_lenet = {task: [] for task in tasks}
all_pred_resnet = {task: [] for task in tasks}
snr_list = []


print("2. Εκτέλεση προβλέψεων (Inference) και για τα δύο μοντέλα...")

with torch.no_grad():
    for images, labels in n_loader:
        images = images.to(device)


        out_lenet = model(images)
        out_resnet = resnet_model(images)


        if 'snr_db' in labels:
            snr_list.extend(labels['snr_db'].cpu().numpy().tolist())

        for i, task in enumerate(tasks):
            all_true[task].extend(labels[task].cpu().numpy().tolist())

            # LeNet Predictions
            _, p_len = torch.max(out_lenet[i], 1)
            all_pred_lenet[task].extend(p_len.cpu().numpy().tolist())

            # ResNet Predictions
            _, p_res = torch.max(out_resnet[i], 1)
            all_pred_resnet[task].extend(p_res.cpu().numpy().tolist())

if len(snr_list) == 0:
    print("-> Τα SNR δεν βρέθηκαν στο test_loader, χρήση τιμών από το CSV...")
    snr_list = df['snr_db'].values[:len(all_true['mod'])]

snr_list = np.array(snr_list)
unique_snrs = np.sort(np.unique(snr_list))

print(f"-> Βρέθηκαν {len(snr_list)} δείγματα.")
print(f"-> Μοναδικά SNRs για τα γραφήματα: {unique_snrs}")


task_titles = {
    'mod': 'Ακρίβεια Modulation ανά SNR',
    'pn': 'Ακρίβεια Phase Noise ανά SNR',
    'iq': 'Ακρίβεια I/Q Imbalance ανά SNR',
    'jam': 'Ακρίβεια Jamming ανά SNR',
    'snr': 'Ακρίβεια Εύρους SNR ανά SNR'
}

print("\n3. Δημιουργία γραφημάτων...")

for task in tasks:
    acc_lenet = []
    acc_resnet = []

    for snr in unique_snrs:
        indices = np.where(snr_list == snr)[0]
        total_at_snr = len(indices)

        if total_at_snr == 0:
            acc_lenet.append(0)
            acc_resnet.append(0)
            continue

        correct_lenet = sum(all_true[task][j] == all_pred_lenet[task][j] for j in indices)
        correct_resnet = sum(all_true[task][j] == all_pred_resnet[task][j] for j in indices)

        acc_lenet.append((correct_lenet / total_at_snr) * 100)
        acc_resnet.append((correct_resnet / total_at_snr) * 100)

    plt.figure(figsize=(10, 6))
    plt.plot(unique_snrs, acc_lenet, marker='o', linewidth=2, label='LeNet', color='blue')
    plt.plot(unique_snrs, acc_resnet, marker='s', linewidth=2, label='ResNet-18', color='orange')

    plt.title(task_titles[task], fontsize=14, fontweight='bold')
    plt.xlabel('Signal-to-Noise Ratio (dB)', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.ylim(0, 105)
    plt.xticks(unique_snrs)
    plt.tight_layout()
    plt.show()